## One-stage anchor-free detector: YOLO11
HW2, notebook 4/5 (Kaggle GPU). Writes `results_04_*.csv`, `curves/`, `final_04.pt` and `submission_04_yolo11.csv` to `/kaggle/working`.

**Needs on Kaggle:** GPU accelerator (T4 / P100), **Internet ON** (pretrained weights are downloaded), inputs: the competition + the output of notebook 01 (or the dataset made from it). No API keys.

**Shared protocol (notebooks 02-05)**
- Data: frames exported by notebook 01 (every 5th frame, ignored regions grayed out). Split **by clip**: 5 GroupKFold folds (grouped by source flight) from notebook 01.
- Cross-validation: every config is trained on `FOLDS_TO_RUN` folds and scored on the held-out clips with the COCO evaluator (ignored regions = `iscrowd`, up to 500 boxes per image). Tables show mean and std over folds.
- Final: the best config (mean mAP) is retrained on all training frames, checked once on the official `val` clips (original resolution, every 5th frame) and used to write the test submission.
- Same metrics everywhere: mAP, AP50, AP_small / medium / large (COCO size buckets), and GPU inference ms per image.

**Model and why**
- **Ultralytics YOLO11s**, COCO-pretrained: anchor-free and by far the fastest to train and run of the four families, which matters for the GPU quota. Built-in mosaic / HSV / flip augmentation is used as is.
- No anchors: each feature-map location predicts a box directly (distribution focal loss on box edges). Positive locations are chosen per object by a **task-aligned assigner** (top-k locations inside the box ranked by classification score and IoU), so no fixed IoU threshold with anchors decides who gets a positive. Tiny objects still only cover few stride-8 cells, which is why resolution matters most.

**Search (budget: 3 configs x folds)**
- `C0_imgsz640`: default settings (also the CNN reference curve for the convergence plot in the `comparisons` notebook).
- `C1_imgsz960`: input resolution 640 -> 960. Search range is limited to these two values by the quota; higher resolution keeps small objects larger on the feature maps.
- `C2_imgsz960_lossgains`: C1 + loss re-weighting `box` 7.5 -> 12, `dfl` 1.5 -> 2 (more weight on localization, which is what suffers for small objects). `cls` is unchanged.

In [1]:
%pip install -q ultralytics

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less GPU memory fragmentation

import os, io, json, time, math, random, shutil, contextlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

SEED = 0
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> turn on a GPU in the session options")

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_all(pattern, depth=5):
    # search attached inputs at increasing nesting depth (Kaggle nests inputs differently per source)
    for d in range(depth):
        hit = sorted(INPUT.glob("/".join(["*"] * d + [pattern])))
        if hit:
            return hit
    return []

def find(pattern):
    hit = find_all(pattern)
    if not hit:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}: check the attached inputs")
    return hit[0]

PREP = find("vd_prep/config.json").parent            # output of notebook 01
CFG = json.load(open(PREP / "config.json"))
TEST_CSV = find("test_images.csv"); ROOT = TEST_CSV.parent
VAL_DIR = find("val/annotations_coco.json").parent
FOLD = CFG["folds"]                                   # train clip -> fold id (GroupKFold by scene, from notebook 01)
METRICS = ["mAP", "AP50", "AP_s", "AP_m", "AP_l", "ms_per_img"]

from ultralytics import YOLO, settings
settings.update({"wandb": False})
DEVICE = "cuda"
EPOCHS, BATCH = 8, 8                   # batch at 640 px (960 px runs use BATCH // 2); effective batch is 64 either way (gradient accumulation)
FOLDS_TO_RUN = [0, 1]                  # folds used for CV (of 5)
VAL_STEP = 5
NB, TAG, MODEL_NAME, MODEL_CLS = "04", "yolo", "YOLO11s", YOLO
COMMON = dict(epochs=EPOCHS, batch=BATCH, workers=4, device=0, seed=SEED, warmup_epochs=1, close_mosaic=2,
              plots=False, exist_ok=True)

GPU: Tesla T4


### Data

In [3]:
def load_gt(path):
    # COCO ground truth; ignored regions (iscrowd=1) are copied to all 8 classes so they ignore detections of any class
    d = json.load(open(path))
    crowd = [a for a in d["annotations"] if a.get("iscrowd")]
    d["annotations"] = [a for a in d["annotations"] if not a.get("iscrowd")]
    nid = max([a["id"] for a in d["annotations"]], default=0) + 1
    for a in crowd:
        for c in range(1, 9):
            d["annotations"].append({**a, "id": nid, "category_id": c, "iscrowd": 1, "area": a["bbox"][2] * a["bbox"][3]})
            nid += 1
    gt = COCO()
    gt.dataset = d
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

def coco_eval(gt, dets, img_ids):
    # standard COCO bbox metrics, up to 500 detections per image (as in the leaderboard)
    if not dets:
        return dict(mAP=0.0, AP50=0.0, AP_s=0.0, AP_m=0.0, AP_l=0.0)
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(gt, gt.loadRes(dets), "bbox")
        E.params.imgIds = list(img_ids)
        E.params.maxDets = [1, 10, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats                                  # s[1], s[3:6] already use maxDets[-1] = 500
    prec = E.eval["precision"][:, :, :, 0, -1]   # [IoU, recall, class] for all sizes at 500 dets (pycocotools' s[0] hardcodes 100)
    prec = prec[prec > -1]
    return dict(mAP=float(prec.mean()) if prec.size else -1.0, AP50=s[1], AP_s=s[3], AP_m=s[4], AP_l=s[5])

def write_submission(dets, path):
    # one row per test image: "class score x y w h ..." (0-based class, pixels, top-left corner, max 500 boxes)
    test = pd.read_csv(TEST_CSV)
    by = {}
    for d in dets:
        by.setdefault(d["image_id"], []).append(d)
    rows = []
    for iid in test.image_id:
        top = sorted(by.get(iid, []), key=lambda d: -d["score"])[:500]
        rows.append(" ".join(f"{d['category_id'] - 1} {d['score']:.4f} " + " ".join(f"{v:.1f}" for v in d["bbox"]) for d in top))
    sub = pd.DataFrame({"image_id": test.image_id, "PredictionString": rows})
    sub.to_csv(path, index=False)
    sample = find_all("sample_submission.csv")      # not shipped with this competition's data: ids come from test_images.csv
    if sample:
        assert list(sub.image_id) == list(pd.read_csv(sample[0]).image_id), "image ids differ from sample_submission.csv"
    print(f"saved {path}: {len(sub)} rows, {sum(map(bool, rows))} with detections, {sum(len(r.split()) // 6 for r in rows)} boxes")

In [4]:
# training frames exported by notebook 01 (ignored regions already grayed out)
TJ = json.load(open(PREP / "train_coco.json"))
IMGS = {i["id"]: i for i in TJ["images"]}
GT_TRAIN = load_gt(PREP / "train_coco.json")

def fold_ids(f):
    # (train ids, held-out ids) for fold f; f=None -> all frames for training
    return ([i for i, m in IMGS.items() if FOLD[m["video"]] != f],
            [i for i, m in IMGS.items() if FOLD[m["video"]] == f])

def copy_parallel(src, dst, workers=16):
    # copy a folder file by file with many threads (on network storage the per-file latency dominates)
    from concurrent.futures import ThreadPoolExecutor
    files = [p for p in Path(src).rglob("*") if p.is_file()]
    def cp(p):
        t = Path(dst) / p.relative_to(src)
        t.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(p, t)
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(cp, files))

# /kaggle/input is network storage (~5 MB/s, ~50 ms per file): copy the training images once to local disk
IMG_ROOT = PREP / "images" / "train"
if str(PREP).startswith("/kaggle/input"):
    local = Path("/tmp/vd_prep/images/train")
    if not (local.parent / ".copied").exists():
        t0 = time.time()
        copy_parallel(IMG_ROOT, local)
        (local.parent / ".copied").touch()
        print(f"copied {sum(1 for _ in local.iterdir())} training images to local disk in {time.time() - t0:.0f} s")
    IMG_ROOT = local

def train_path(i):
    return IMG_ROOT / IMGS[i]["file_name"]

# official val: raw frames at original resolution, every VAL_STEP-th frame of each clip (for speed)
VAL_GT = load_gt(VAL_DIR / "annotations_coco.json")
_idx = {(p.parent.name, p.name): p for p in (VAL_DIR / "images").rglob("*.jpg")}
_v = pd.DataFrame(VAL_GT.dataset["images"])
_v = _v[(_v.groupby("video").frame_id.rank(method="first").astype(int) - 1) % VAL_STEP == 0]
VAL_ITEMS = [(int(r.id), _idx[(r.video, Path(r.file_name).name)]) for r in _v.itertuples()]
VAL_IDS = [i for i, _ in VAL_ITEMS]
TEST_ITEMS = [(r.image_id, ROOT / r.file_name) for r in pd.read_csv(TEST_CSV).itertuples()]
print(f"{len(IMGS)} train frames | {len(VAL_ITEMS)} val frames | {len(TEST_ITEMS)} test frames")

copied 4868 training images to local disk in 10 s
4868 train frames | 571 val frames | 3811 test frames


In [5]:
import sys, subprocess

DEVICE_ID = globals().get("DEVICE_ID", 0)
# Ultralytics finds labels by swapping images -> labels in the path, and writes a .cache next to them
IMG_DIR = IMG_ROOT                                          # local copy of the training images (data cell)
LBL_DIR = IMG_DIR.parent.parent / "labels" / "train"
if not LBL_DIR.exists():
    copy_parallel(PREP / "labels" / "train", LBL_DIR)
JOB_DIR = Path("/tmp/yolo_jobs") if str(WORK).startswith("/kaggle") else WORK / "yolo_jobs"   # image lists, yamls, job files
JOB_DIR.mkdir(parents=True, exist_ok=True)

def write_data(name, tr_ids, va_ids):
    # Ultralytics data yaml with explicit image lists, so folds are just different lists
    for part, ids in (("train", tr_ids), ("val", va_ids)):
        (JOB_DIR / f"{name}_{part}.txt").write_text("\n".join(str(IMG_DIR / IMGS[i]["file_name"]) for i in ids))
    q = lambda x: json.dumps(str(x))                        # quoted path, safe for Windows backslashes
    p = JOB_DIR / f"{name}.yaml"
    p.write_text(f"path: {q(JOB_DIR)}\ntrain: {q(JOB_DIR / (name + '_train.txt'))}\nval: {q(JOB_DIR / (name + '_val.txt'))}\nnames: {json.dumps(CFG['classes'])}\n")
    return str(p)

# Every training and every prediction runs in its own short-lived Python process. When it exits, all of its
# GPU memory is released, so nothing piles up between runs (inside one process it did, and the GPU ran out).
WORKER = JOB_DIR / "ultra_worker.py"
WORKER.write_text("""
import json, sys
job = json.load(open(sys.argv[1]))
import torch
from ultralytics import YOLO, RTDETR, settings
settings.update({"wandb": False})
Cls = {"YOLO": YOLO, "RTDETR": RTDETR}[job["cls"]]
out = {}
if job["kind"] == "train":
    model = Cls(job["build"])
    if job.get("load"):
        model = model.load(job["load"])                     # built from a yaml: transfer the pretrained weights
    model.train(**job["kwargs"])
else:
    from ultralytics.utils import DEFAULT_CFG_DICT
    kw = dict(job["kwargs"])
    if job.get("fp16"):                                     # newer versions renamed half=True to quantize=16
        kw.update({"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True})
    model = Cls(job["weights"])
    items = json.load(open(job["items"]))
    dets, ms, CHUNK = [], [], 16
    # small chunks: Ultralytics 8.4 loads a whole list of paths as ONE batch (ignores `batch`) -> out of memory
    for k in range(0, len(items), CHUNK):
        chunk = items[k:k + CHUNK]
        for (iid, _), r in zip(chunk, model.predict([p for _, p in chunk], stream=True, **kw)):
            b = r.boxes
            xywh = b.xyxy.cpu().numpy()
            xywh[:, 2:] -= xywh[:, :2]
            for box, s, c in zip(xywh.tolist(), b.conf.tolist(), b.cls.tolist()):
                dets.append([iid, int(c) + 1, *[round(v, 2) for v in box], round(s, 5)])
            ms.append(r.speed["inference"] + r.speed["postprocess"])
    out["dets"], out["ms"] = dets, sum(ms) / max(len(ms), 1)
if torch.cuda.is_available():
    out["peak_gb"] = torch.cuda.max_memory_reserved() / 1e9
json.dump(out, open(job["out"], "w"))
""")

def run_worker(job, tag):
    # runs one job in a fresh process, prints its log here, returns its result
    job = {**job, "out": str(JOB_DIR / f"{tag}.out.json")}
    jp = JOB_DIR / f"{tag}.job.json"
    jp.write_text(json.dumps(job))
    proc = subprocess.Popen([sys.executable, str(WORKER), str(jp)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{tag}: the worker process failed, see its log above")
    return json.loads(Path(job["out"]).read_text())

def fit(spec, name, tr_ids, va_ids, **hp):
    kw = {"data": write_data(name, tr_ids, va_ids), "project": str(WORK / "runs"), "name": name, **{**COMMON, **hp}}
    res = run_worker({**spec, "kind": "train", "kwargs": kw}, name)
    (WORK / "curves").mkdir(exist_ok=True)
    shutil.copy(WORK / "runs" / name / "results.csv", WORK / "curves" / f"{TAG}_{name}.csv")     # per-epoch val curve
    # last epoch, not "best": best is picked on the held-out fold
    return WORK / "runs" / name / "weights" / "last.pt", res.get("peak_gb", float("nan"))

def detect(weights, items, imgsz):
    # returns COCO-style detections in original pixel coordinates and model ms per image (inference + postprocess)
    tag = f"predict_{Path(weights).parent.parent.name}_{len(items)}"
    ip = JOB_DIR / f"{tag}.items.json"
    ip.write_text(json.dumps([[i, str(p)] for i, p in items]))
    kw = dict(imgsz=imgsz, conf=0.001, iou=0.6, max_det=500, batch=4, verbose=False, device=DEVICE_ID)
    res = run_worker({"cls": MODEL_CLS.__name__, "kind": "predict", "weights": str(weights), "items": str(ip),
                      "kwargs": kw, "fp16": True}, tag)
    dets = [{"image_id": d[0], "category_id": d[1], "bbox": d[2:6], "score": d[6]} for d in res["dets"]]
    return dets, res["ms"]

def eval_fold(weights, ids, imgsz):
    dets, ms = detect(weights, [(i, IMG_DIR / IMGS[i]["file_name"]) for i in ids], imgsz)
    return {**coco_eval(GT_TRAIN, dets, ids), "ms_per_img": ms}

### Cross-validation of the three configs

In [6]:
CONFIGS = {
    "C0_imgsz640":           dict(imgsz=640),
    "C1_imgsz960":           dict(imgsz=960),
    "C2_imgsz960_lossgains": dict(imgsz=960, box=12.0, dfl=2.0),     # defaults: box 7.5, cls 0.5, dfl 1.5
}
make_model = lambda cfg: {"cls": "YOLO", "build": "yolo11s.pt"}     # built inside the worker process
# 960 px runs use half the batch: same effective batch (Ultralytics accumulates gradients up to 64 images)
hp_of = lambda cfg: {**cfg, "batch": BATCH if cfg["imgsz"] <= 640 else BATCH // 2}

### GPU memory check (~3 min)
One epoch on the 64 most crowded frames with the heaviest config (mosaics of crowded frames are the worst case for GPU memory), then a prediction on them at the same image size.
If memory is too tight the notebook stops here, not after hours of training. Then lower `BATCH` in the setup cell.

In [7]:
n_obj = {}
for a in TJ["annotations"]:
    if not a.get("iscrowd"):
        n_obj[a["image_id"]] = n_obj.get(a["image_id"], 0) + 1
dense = sorted(IMGS, key=lambda i: -n_obj.get(i, 0))[:64]
heavy = list(CONFIGS.values())[-1]
kw = {"data": write_data("memcheck", dense, dense[:16]), "project": str(WORK / "runs"), "name": "memcheck",
      **{**COMMON, **hp_of(heavy), "epochs": 1, "warmup_epochs": 0, "close_mosaic": 0}}
res = run_worker({**make_model(heavy), "kind": "train", "kwargs": kw}, "memcheck")
# the evaluation path too, on the same crowded frames at the largest image size
mc_dets, _ = detect(WORK / "runs" / "memcheck" / "weights" / "last.pt", [(i, IMG_DIR / IMGS[i]["file_name"]) for i in dense], heavy["imgsz"])
print(f"prediction check: {len(dense)} frames -> {len(mc_dets)} boxes")
shutil.rmtree(WORK / "runs" / "memcheck", ignore_errors=True)
peak, total = res.get("peak_gb", 0.0), torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"peak GPU memory {peak:.1f} of {total:.1f} GB")
assert peak < 0.85 * total, "too close to the GPU memory limit: lower BATCH in the setup cell and run again"

1 epochs completed in 0.002 hours.
prediction check: 64 frames -> 32000 boxes
peak GPU memory 3.2 of 15.6 GB


### Cross-validation runs

In [8]:
# every config is trained on the same folds (cross-validation over clips) and scored on the held-out clips.
# Results are saved after every run; runs already in results_{NB}_folds.csv (here, or in an attached
# earlier version's output) are skipped, so a crash does not cost the finished runs again.
prev = [p for p in [WORK / f"results_{NB}_folds.csv"] if p.exists()] + find_all(f"results_{NB}_folds.csv")
rows = pd.read_csv(prev[0]).to_dict("records") if prev else []
done = {(r["config"], int(r["fold"])) for r in rows}
if done:
    print("reusing finished runs from", prev[0], ":", sorted(done))
    pd.DataFrame(rows).to_csv(WORK / f"results_{NB}_folds.csv", index=False)     # keep them in this version's output too
(WORK / "curves").mkdir(exist_ok=True)
for name, cfg in CONFIGS.items():
    for f in FOLDS_TO_RUN:
        if (name, f) in done:
            for p in find_all(f"curves/{TAG}_{name}_f{f}.csv"):        # keep its training curve for notebook 06
                if not (WORK / "curves" / p.name).exists():
                    shutil.copy(p, WORK / "curves" / p.name)
            continue
        tr_ids, va_ids = fold_ids(f)
        t0 = time.time()
        run = f"{name}_f{f}"
        trained = [p for p in [WORK / "runs" / run / "weights" / "last.pt"] if p.exists()] + find_all(f"runs/{run}/weights/last.pt")
        curve = [p for p in [WORK / "runs" / run / "results.csv"] if p.exists()] + find_all(f"runs/{run}/results.csv")
        if trained and curve:                                   # trained in an earlier, interrupted version: only evaluate
            print("reusing trained weights", trained[0])
            w, peak = trained[0], float("nan")
            if curve[0] != WORK / "curves" / f"{TAG}_{run}.csv":
                shutil.copy(curve[0], WORK / "curves" / f"{TAG}_{run}.csv")
        else:
            w, peak = fit(make_model(cfg), run, tr_ids, va_ids, **hp_of(cfg))
        m = eval_fold(w, va_ids, cfg["imgsz"])
        rows.append({"config": name, "fold": f, **m, "train_min": (time.time() - t0) / 60, "peak_gb": peak})
        pd.DataFrame(rows).to_csv(WORK / f"results_{NB}_folds.csv", index=False)
        print(name, "fold", f, {k: round(v, 4) for k, v in m.items()}, f"| peak GPU memory of the training: {peak:.1f} GB")
folds_df = pd.DataFrame(rows)
folds_df = folds_df[folds_df.config.isin(CONFIGS) & folds_df.fold.isin(FOLDS_TO_RUN)]
display(folds_df.groupby("config")[METRICS].agg(["mean", "std"]).round(4))

reusing finished runs from /kaggle/input/datasets/vivtrea/results-04-folds-csv/results_04_folds.csv : [('C0_imgsz640', 0), ('C0_imgsz640', 1)]
reusing trained weights /kaggle/input/notebooks/vivtrea/04-yolo11/runs/C1_imgsz960_f0/weights/last.pt
C1_imgsz960 fold 0 {'mAP': 0.203, 'AP50': np.float64(0.3836), 'AP_s': np.float64(0.08), 'AP_m': np.float64(0.2658), 'AP_l': np.float64(0.4741), 'ms_per_img': 6.4068} | peak GPU memory of the training: nan GB
8 epochs completed in 0.313 hours.
C1_imgsz960 fold 1 {'mAP': 0.1683, 'AP50': np.float64(0.3246), 'AP_s': np.float64(0.0655), 'AP_m': np.float64(0.2482), 'AP_l': np.float64(0.4492), 'ms_per_img': 6.9199} | peak GPU memory of the training: 3.1 GB
8 epochs completed in 0.301 hours.
C2_imgsz960_lossgains fold 0 {'mAP': 0.2018, 'AP50': np.float64(0.3883), 'AP_s': np.float64(0.0835), 'AP_m': np.float64(0.2678), 'AP_l': np.float64(0.4662), 'ms_per_img': 6.8313} | peak GPU memory of the training: 3.1 GB
8 epochs completed in 0.314 hours.
C2_imgsz96

mAP            AP50            AP_s            AP_m  \
                         mean     std    mean     std    mean     std    mean   
config                                                                          
C0_imgsz640            0.1538  0.0188  0.2911  0.0322  0.0494  0.0027  0.2065   
C1_imgsz960            0.1857  0.0245  0.3541  0.0417  0.0727  0.0103  0.2570   
C2_imgsz960_lossgains  0.1875  0.0202  0.3595  0.0407  0.0753  0.0116  0.2593   

                                 AP_l         ms_per_img          
                          std    mean     std       mean     std  
config                                                            
C0_imgsz640            0.0019  0.4213  0.0539     4.1516  0.0255  
C1_imgsz960            0.0125  0.4617  0.0176     6.6634  0.3628  
C2_imgsz960_lossgains  0.0120  0.4375  0.0407     6.8372  0.0084

**Observations:**
- Resolution is the main lever: 640 -> 960 px raises CV mAP 0.154 -> 0.186 and AP_small 0.049 -> 0.073 (+48 %), at 6.7 vs 4.2 ms per image.
- Higher box / dfl loss weights add only +0.002 mAP and +0.003 AP_small, while AP_large drops 0.462 -> 0.438. That is within the fold std (~0.02), so it is not a clear improvement. C2 was still selected by mean CV mAP.
- The two folds differ by ~0.03 mAP for every config, which is bigger than the difference between C1 and C2.

### Final model: retrain on all frames, check on official val

In [9]:
means = folds_df.groupby("config")[METRICS].mean()
best = means.mAP.idxmax()
print("best config by CV mAP:", best)
tr_all, _ = fold_ids(None)
w, _ = fit(make_model(CONFIGS[best]), "final", tr_all, fold_ids(0)[1][:200], **hp_of(CONFIGS[best]))   # 200 held-out frames only feed the training log
shutil.copy(w, WORK / f"final_{NB}.pt")

val_dets, val_ms = detect(w, VAL_ITEMS, CONFIGS[best]["imgsz"])
final_metrics = {**coco_eval(VAL_GT, val_dets, VAL_IDS), "ms_per_img": val_ms}
display(pd.Series(final_metrics).round(4).to_frame("official val"))

out = means.reset_index().assign(split="cv")
out = pd.concat([out, pd.DataFrame([{"config": best + " (final)", **final_metrics, "split": "val"}])], ignore_index=True)
out.insert(0, "model", MODEL_NAME)
out.to_csv(WORK / f"results_{NB}_summary.csv", index=False)

best config by CV mAP: C2_imgsz960_lossgains
8 epochs completed in 0.354 hours.


,official val
mAP,0.1699
AP50,0.3429
AP_s,0.0976
AP_m,0.1655
AP_l,0.3009
ms_per_img,7.9346


**Observations:**
- Final model on the official val: mAP 0.170, AP50 0.343, AP_small 0.098, 7.9 ms per image (lower than CV 0.188: different clips, and val has a 4K clip whose objects become tiny at 960 px).
- Without anchors, the task-aligned assigner picks the top-k grid locations inside each box, ranked by classification score x IoU. Every object gets positives regardless of its shape or size, but a 10-15 px object covers only 1-2 stride-8 cells, and in dense scenes neighbouring objects compete for the same cells. That is why input resolution helped much more than the loss weights.
- Compared with RetinaNet at the same CV folds: AP_small 0.075 vs 0.026, AP_medium 0.259 vs 0.188, AP_large 0.438 vs 0.438, and ~6x faster (6.8 vs 40 ms). The gain comes almost entirely from small and medium objects. The full comparison is in the `comparisons` notebook.

### Test predictions and submission

In [10]:
test_dets, _ = detect(w, TEST_ITEMS, CONFIGS[best]["imgsz"])
write_submission(test_dets, WORK / "submission_04_yolo11.csv")

saved /kaggle/working/submission_04_yolo11.csv: 3811 rows, 3811 with detections, 1199634 boxes
